In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

%matplotlib inline


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
data = pd.read_csv("/Users/akashdeepsangwan/Desktop/ML/ML/GitHub-Collab/Machine-Learning/Isolation_Forest/blood_cell_anomaly_detection.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/Users/akashdeepsangwan/Desktop/ML/ML/GitHub-Collab/Machine-Learning/Isolation_Forest/blood_cell_anomaly_detection.csv'

In [ ]:
data.head(10)

,cell_id,cell_type,anomaly_label,disease_category,cell_diameter_um,nucleus_area_pct,chromatin_density,cytoplasm_ratio,circularity,eccentricity,...,mcv_fl,mchc_g_dl,dataset_source,staining_protocol,microscope_model,magnification_x,image_resolution_px,cytodiffusion_anomaly_score,cytodiffusion_classification_confidence,labeller_confidence_score
0,CELL_005371,Hypersegmented_Neutrophil,1,Infection,15.18,58.8,0.542,0.301,0.563,0.529,...,85.5,31.4,CytoData,Giemsa,Zeiss_Axio,100,224,0.7649,0.5726,0.5670
1,CELL_005300,Hypersegmented_Neutrophil,1,Infection,16.47,73.6,0.583,0.365,0.859,0.443,...,92.5,35.0,PBC_Dataset,Wright,Zeiss_Axio,100,224,0.8472,0.7150,0.7273
2,CELL_000200,Neutrophil,0,Normal_WBC,13.41,55.5,0.448,0.376,0.781,0.407,...,76.3,33.0,CytoData,Wright,Leica_DM2000,100,512,0.0313,0.9225,0.9623
3,CELL_003269,Normal_RBC,0,Normal_RBC,7.36,0.0,0.000,1.000,0.880,0.167,...,92.3,32.5,CytoData,Wright,Leica_DM2000,100,512,0.1293,0.9180,0.8652
4,CELL_003505,Normal_RBC,0,Normal_RBC,7.53,0.0,0.000,1.000,1.000,0.158,...,83.9,33.4,CytoData,Wright,Olympus_BX51,100,224,0.1418,0.9697,0.8898
5,CELL_005586,Reactive_Lymphocyte,1,Infection,14.97,57.8,0.730,0.188,0.777,0.351,...,93.7,32.1,PBC_Dataset,Giemsa,Leica_DM2000,100,224,0.7682,0.6462,1.0000
6,CELL_002345,Monocyte,0,Normal_WBC,15.63,63.3,0.503,0.571,0.559,0.312,...,80.9,34.1,PBC_Dataset,Giemsa,Olympus_BX51,60,360,0.0382,0.9248,0.9460
7,CELL_001612,Lymphocyte,0,Normal_WBC,9.15,78.3,0.790,0.130,0.820,0.045,...,88.5,33.9,Raabin_WBC,Giemsa,Olympus_BX51,60,512,0.1254,0.9813,0.9028
8,CELL_005756,Smudge_Cell,1,Artefact,10.10,92.9,0.685,0.050,0.288,0.761,...,80.6,32.0,Raabin_WBC,May_Grunwald_Giemsa,Leica_DM2000,100,512,0.9006,0.7303,0.7140
9,CELL_002202,Monocyte,0,Normal_WBC,16.97,64.1,0.454,0.449,0.785,0.575,...,66.3,32.8,Raabin_WBC,Wright,Olympus_BX51,100,360,0.0996,0.8527,0.9048


## Step 1: inspect the table before dropping anything
- Isolation Forest only cares about the numbers it splits on.
- IDs, labels, and leftover model scores are not morphology. If we leave them in, the detector cheats instead of cleaning the data.
- We look at shape, missing values, and which columns already *are* the anomaly label.

In [4]:
# basic health check: rows, columns, types, missing values, class balance
print("shape:", data.shape)
print("\ndtypes:\n", data.dtypes)
print("\nmissing values:\n", data.isna().sum())
print("\nanomaly_label counts:\n", data["anomaly_label"].value_counts())


shape: (5880, 36)

dtypes:
 cell_id                                     object
cell_type                                   object
anomaly_label                                int64
disease_category                            object
cell_diameter_um                           float64
nucleus_area_pct                           float64
chromatin_density                          float64
cytoplasm_ratio                            float64
circularity                                float64
eccentricity                               float64
granularity_score                          float64
lobularity_score                           float64
membrane_smoothness                        float64
cell_area_px                                 int64
perimeter_px                                 int64
mean_r                                       int64
mean_g                                       int64
mean_b                                       int64
stain_intensity                            float64
pat

## Step 2: find columns that leak the answer
- `cell_id` is just a name for the row. Drop it.
- `anomaly_label` is the target. Keep it as `y` for scoring permutation importance, never as a feature.
- `cell_type` and `disease_category` line up perfectly with the label (every abnormal type is always 1).
- `cytodiffusion_*` and `labeller_confidence_score` are already another model's output, not a measured cell trait.
- Those leaky columns would look "important" in permutation importance, but they would not help us *clean* the raw measurements.

In [5]:
# if a column already IS the label, every row of that type will share the same anomaly_label
print(pd.crosstab(data["cell_type"], data["anomaly_label"]))
print()
print(pd.crosstab(data["disease_category"], data["anomaly_label"]))
print()

# correlation of leftover model scores with the label (close to 1 = leakage)
print(
    data[
        [
            "cytodiffusion_anomaly_score",
            "cytodiffusion_classification_confidence",
            "labeller_confidence_score",
            "anomaly_label",
        ]
    ].corr()["anomaly_label"]
)


anomaly_label                 0    1
cell_type                           
Artefact                      0   80
Basophil                    150    0
Blast_Cell                    0  280
Elliptocyte                   0  200
Eosinophil                  300    0
Hypersegmented_Neutrophil     0  160
Lymphocyte                  850    0
Monocyte                    400    0
Neutrophil                 1100    0
Normal_RBC                  900    0
Platelet                    300    0
Prolymphocyte                 0  180
Reactive_Lymphocyte           0  150
Schistocyte                   0  170
Sickle_Cell                   0  140
Smudge_Cell                   0  100
Spherocyte                    0  150
Target_Cell                   0  130
Toxic_Granulation             0  140

anomaly_label          0    1
disease_category             
Anemia                 0  650
Artefact               0  180
Infection              0  450
Leukemia               0  460
Normal_Platelet      300    0
Normal_RBC  

## Step 3: drop IDs and leaky columns, keep the rest for the detector
- Isolation Forest isolates a point by random splits. A point is "anomalous" if it becomes alone after only a few splits.
- If we leave in an ID, the label, or another model's score, one cheap split already isolates the row. The forest never has to read circularity, nucleus size, stain, etc.
- That looks like high accuracy against `anomaly_label`, but we extracted **no new information** from the measurements. Dropping these columns forces the trees to isolate using the actual cell data.
- `X` = what the forest may split on. `y` = `anomaly_label`, used later only to *score* permutation importance.

In [6]:
data.head(10
)

,cell_id,cell_type,anomaly_label,disease_category,cell_diameter_um,nucleus_area_pct,chromatin_density,cytoplasm_ratio,circularity,eccentricity,...,mcv_fl,mchc_g_dl,dataset_source,staining_protocol,microscope_model,magnification_x,image_resolution_px,cytodiffusion_anomaly_score,cytodiffusion_classification_confidence,labeller_confidence_score
0,CELL_005371,Hypersegmented_Neutrophil,1,Infection,15.18,58.8,0.542,0.301,0.563,0.529,...,85.5,31.4,CytoData,Giemsa,Zeiss_Axio,100,224,0.7649,0.5726,0.5670
1,CELL_005300,Hypersegmented_Neutrophil,1,Infection,16.47,73.6,0.583,0.365,0.859,0.443,...,92.5,35.0,PBC_Dataset,Wright,Zeiss_Axio,100,224,0.8472,0.7150,0.7273
2,CELL_000200,Neutrophil,0,Normal_WBC,13.41,55.5,0.448,0.376,0.781,0.407,...,76.3,33.0,CytoData,Wright,Leica_DM2000,100,512,0.0313,0.9225,0.9623
3,CELL_003269,Normal_RBC,0,Normal_RBC,7.36,0.0,0.000,1.000,0.880,0.167,...,92.3,32.5,CytoData,Wright,Leica_DM2000,100,512,0.1293,0.9180,0.8652
4,CELL_003505,Normal_RBC,0,Normal_RBC,7.53,0.0,0.000,1.000,1.000,0.158,...,83.9,33.4,CytoData,Wright,Olympus_BX51,100,224,0.1418,0.9697,0.8898
5,CELL_005586,Reactive_Lymphocyte,1,Infection,14.97,57.8,0.730,0.188,0.777,0.351,...,93.7,32.1,PBC_Dataset,Giemsa,Leica_DM2000,100,224,0.7682,0.6462,1.0000
6,CELL_002345,Monocyte,0,Normal_WBC,15.63,63.3,0.503,0.571,0.559,0.312,...,80.9,34.1,PBC_Dataset,Giemsa,Olympus_BX51,60,360,0.0382,0.9248,0.9460
7,CELL_001612,Lymphocyte,0,Normal_WBC,9.15,78.3,0.790,0.130,0.820,0.045,...,88.5,33.9,Raabin_WBC,Giemsa,Olympus_BX51,60,512,0.1254,0.9813,0.9028
8,CELL_005756,Smudge_Cell,1,Artefact,10.10,92.9,0.685,0.050,0.288,0.761,...,80.6,32.0,Raabin_WBC,May_Grunwald_Giemsa,Leica_DM2000,100,512,0.9006,0.7303,0.7140
9,CELL_002202,Monocyte,0,Normal_WBC,16.97,64.1,0.454,0.449,0.785,0.575,...,66.3,32.8,Raabin_WBC,Wright,Olympus_BX51,100,360,0.0996,0.8527,0.9048


In [ ]:
# columns Isolation Forest must not train on
drop_cols = [
    "cell_id",
    "anomaly_label",
    "cell_type",
    "disease_category",
    "cytodiffusion_anomaly_score",
    "cytodiffusion_classification_confidence",
    "labeller_confidence_score",
]

y = data["anomaly_label"].copy()          # ground truth for scoring only
X = data.drop(columns=drop_cols).copy()   # remaining raw measurements / metadata

print("features going into the detector:\n", list(X.columns))
print("\nX shape:", X.shape)


## Step 4: turn leftover text columns into numbers
- Isolation Forest can only split on numeric columns.
- `patient_age_group`, `patient_sex`, `dataset_source`, `staining_protocol`, `microscope_model` are still strings.
- Label encoding keeps one column per feature, which makes permutation importance easier to read than one-hot dummies.

In [ ]:
# Isolation Forest needs numbers, so encode every object column in place
X_encoded = X.copy()
label_encoders = {}

for col in X_encoded.select_dtypes(include="object").columns:
    encoder = LabelEncoder()
    X_encoded[col] = encoder.fit_transform(X_encoded[col])
    label_encoders[col] = encoder   # keep the mapping if we need to decode later

print("categorical columns encoded:", list(label_encoders.keys()))
X_encoded.head()


## Step 5: fit Isolation Forest, then permutation importance
- Isolation Forest is unsupervised: it only fits on `X_encoded`, not on `y`.
- Permutation importance then asks: if I shuffle one feature, how much worse does the detector rank real anomalies?
- We score with ROC-AUC of the anomaly scores vs `y`. A drop in AUC means that feature was useful.
- Features with importance near 0 (or negative) are noise for this detector, so we drop them.

In [ ]:
# Isolation Forest does not use y. 'auto' keeps the fit unsupervised
# (contamination only changes the later predict() cutoff, not these scores)
iso = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42,
)
iso.fit(X_encoded)

# sklearn IF: score_samples is higher for normal points, so we negate it
def anomaly_auc(estimator, X, y_true):
    anomaly_score = -estimator.score_samples(X)
    return roc_auc_score(y_true, anomaly_score)

print("AUC on the unshuffled features:", anomaly_auc(iso, X_encoded, y))


In [ ]:
# shuffle each feature, re-score the fitted forest, repeat a few times for a stable mean
perm = permutation_importance(
    iso,
    X_encoded,
    y,
    scoring=anomaly_auc,
    n_repeats=8,
    random_state=42,
)

# table: high mean = detector needs this column; ~0 = safe to drop
importance_df = pd.DataFrame(
    {
        "feature": X_encoded.columns,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    }
).sort_values("importance_mean", ascending=False)

importance_df


In [ ]:
# horizontal bar: error bars are the std across the 8 shuffles
plt.figure(figsize=(8, 10))
plt.barh(
    importance_df["feature"],
    importance_df["importance_mean"],
    xerr=importance_df["importance_std"],
)
plt.gca().invert_yaxis()
plt.xlabel("drop in AUC after shuffling the feature")
plt.title("Permutation importance (Isolation Forest)")
plt.tight_layout()
plt.show()


## Step 6: drop features the detector does not use
- Importance near 0: shuffling that column did not change the anomaly ranking.
- Negative importance: shuffling it *helped* a little, so it was noise.
- Keep only features with mean importance above 0, then the Isolation Forest can clean the table on a tighter feature set.

In [ ]:
# keep a feature only if shuffling it actually hurt the detector
importance_threshold = 0.0
keep_features = importance_df.loc[
    importance_df["importance_mean"] > importance_threshold, "feature"
].tolist()
drop_features = importance_df.loc[
    importance_df["importance_mean"] <= importance_threshold, "feature"
].tolist()

print("kept:", keep_features)
print("\ndropped as unused:", drop_features)

# cleaned feature matrix for the next Isolation Forest pass
X_selected = X_encoded[keep_features].copy()
print("\nX_selected shape:", X_selected.shape)
X_selected.head()


In [8]:
for i in range(10):
    print(i, "\n")

0 

1 

2 

3 

4 

5 

6 

7 

8 

9 



## Pause
- `X_selected` is the feature set the Isolation Forest actually uses.
- Next step: refit Isolation Forest on `X_selected` and use its scores to flag rows to clean / drop.
- Re-run from the import cell so the new libraries load before permutation importance.